# Data Scrapping

In [70]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

urls = ["https://en.wikipedia.org/wiki/1930_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1934_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1938_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1950_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1954_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1958_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1962_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1966_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1970_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1974_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1978_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1982_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1986_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1990_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1994_FIFA_World_Cup", "https://en.wikipedia.org/wiki/1998_FIFA_World_Cup", "https://en.wikipedia.org/wiki/2002_FIFA_World_Cup", "https://en.wikipedia.org/wiki/2006_FIFA_World_Cup", "https://en.wikipedia.org/wiki/2010_FIFA_World_Cup", "https://en.wikipedia.org/wiki/2014_FIFA_World_Cup", "https://en.wikipedia.org/wiki/2018_FIFA_World_Cup", "https://en.wikipedia.org/wiki/2022_FIFA_World_Cup"]
years = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]

def parse_wiki_matches(html, year):
    soup = BeautifulSoup(html, "html.parser")
    matches = []
    current_stage = "Group Stage"
    current_group = None

    for tag in soup.find_all(["h2", "h3", "div"]):
        if tag.name in ["h2", "h3"]:
            text = tag.get_text(strip=True)
            if "Round of 16" in text:
                current_stage = "Round of 16"
                current_group = None
            elif "Quarter" in text:
                current_stage = "Quarterfinals"
                current_group = None
            elif "Semi" in text:
                current_stage = "Semifinals"
                current_group = None
            elif "Final" in text and "third" not in text.lower():
                current_stage = "Final"
                current_group = None
            elif "Match for third place" in text:
                current_stage = "Third Place Match"
                current_group = None
            elif "Group" in text:
                current_stage = "Group Stage"
                current_group = text.strip()

        if tag.name == "div" and "footballbox" in tag.get("class", []):
            try:
                home_team = tag.select_one(".fhome span[itemprop='name']").text.strip()
                away_team = tag.select_one(".faway span[itemprop='name']").text.strip()
                score_tag = tag.select_one("th.fscore")
                score_text = None

                if score_tag:
                    score_text = " ".join(score_tag.stripped_strings)

                home_score, away_score, score_info = None, None, None

                if score_text:
                    match = re.match(r"(\d+)[–-](\d+)\s*(?:\((.+)\))?", score_text)
                    if match:
                        home_score = int(match.group(1))
                        away_score = int(match.group(2))
                        score_info = match.group(3) if match.group(3) else None
                matches.append({"year": year, "stage": current_stage, "group": current_group, "home_team": home_team, "away_team": away_team, "home_score": home_score, "away_score": away_score, "score_info": score_info})

            except Exception:
                continue
    return matches

all_matches = []
for url, year in zip(urls, years):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        matches = parse_wiki_matches(response.text, year)
        all_matches.extend(matches)
    else:
        print("Bad URL")

df = pd.DataFrame(all_matches)
df.to_csv("world_cup_matches_scrape.csv", index=False)

In [58]:
import time

def parse_world_cup_html(html_content, source_url, year):
    soup = BeautifulSoup(html_content, "html.parser")
    pre = soup.find("pre")
    if not pre:
        return []
    
    text = pre.get_text("\n")
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    rows = []
    current_group = None
    current_stage = "Group Stage"
    current_odds = None

    for i, line in enumerate(lines):
        if "Eightfinals" in line:
            current_stage = "Round of 16"
            continue
        elif "1/8 Finals" in line:
            current_stage = "Round of 16"
            continue
        elif "Second Round" in line:
            current_stage = "Round of 16"
            continue
        elif "Quarterfinals" in line:
            current_stage = "Quarterfinals"
            continue
        elif "Quarter Finals" in line:
            current_stage = "Quarterfinals"
            continue
        elif "Quarter finals" in line:
            current_stage = "Quarterfinals"
            continue
        elif "Semifinals" in line:
            current_stage = "Semifinals"
            continue
        elif "Group 1" in line:
            current_stage = "Group 1"
            continue
        elif "Group 2" in line:
            current_stage = "Group 2"
            continue
        elif "Group 3" in line:
            current_stage = "Group 3"
            continue
        elif "Group 4" in line:
            current_stage = "Group 4"
            continue
        elif "Third Place Match" in line:
            current_stage = "Third Place Match"
            continue
        elif "Third place match" in line:
            current_stage = "Third Place Match"
            continue
        elif "MATCH FOR THIRD PLACE" in line:
            current_stage = "Third Place Match"
            continue
        elif "THIRD PLACE MATCH" in line:
            current_stage = "Third Place Match"
            continue
        elif "Final Pool" in line:
            current_stage = "Final Pool"
            continue
        elif "FINAL" in line:
            current_stage = "Final"
            continue
        elif "Final" in line:
            current_stage = "Final"
            continue
        
        # -------------------------
        # Detect groups
        # -------------------------
        group_match = re.search(r'Group [A-L]', line)
        if group_match:
            current_group = group_match.group()
            current_stage = "Group Stage"

        match_match = re.search(r'([A-Z]{3})\s*-\s*([A-Z]{3})\s*(\d+:\d+)', line)
        if match_match:
            # Look for odds in the same line
            odds_match = re.search(r'\(([~+\-]?\d+)\)', line)
            if odds_match:
                odds_for_match = odds_match.group(1)
            else:
                odds_for_match = current_odds  # fallback to previous odds if present
    
            rows.append({
                "year": year,
                "stage": current_stage,
                "group": current_group if current_stage == "Group Stage" else None,
                "team1": match_match.group(1),
                "team2": match_match.group(2),
                "score": match_match.group(3),
                "odds": odds_for_match,
                "source_url": source_url
            })

            # Reset odds after using it
            current_odds = None
            continue
    
        # -------------------------
        # Detect odds lines (ignore sent off / booked lines)
        # -------------------------
        if "sent off" not in line_lower and "sent-off" not in line.lower() and "booked" not in line_lower:
            odds_match = re.search(r'\(([~+\-]?\d+)\)', line)
            if odds_match:
                current_odds = odds_match.group(1)
                
                # Reset odds after using it
                current_odds = None
        
        return rows


# =========================
# YOUR DATA
# =========================
urls = [
    "https://www.rsssf.org/tables/30full.html",
    "https://www.rsssf.org/tables/34full.html",
    "https://www.rsssf.org/tables/38full.html",
    "https://www.rsssf.org/tables/50full.html",
    "https://www.rsssf.org/tables/54full.html",
    "https://www.rsssf.org/tables/58full.html",
    "https://www.rsssf.org/tables/62full.html",
    "https://www.rsssf.org/tables/66full.html",
    "https://www.rsssf.org/tables/70full.html",
    "https://www.rsssf.org/tables/74full.html",
    "https://www.rsssf.org/tables/78full.html",
    "https://www.rsssf.org/tables/82full.html",
    "https://www.rsssf.org/tables/86full.html",
    "https://www.rsssf.org/tables/90full.html",
    "https://www.rsssf.org/tables/94full.html",
    "https://www.rsssf.org/tables/98full.html",
    "https://www.rsssf.org/tables/2002full.html",
    "https://www.rsssf.org/tables/2006full.html",
    "https://www.rsssf.org/tables/2010full.html",
    "https://www.rsssf.org/tables/2014full.html"
]

years = [
    1930,
    1934,
    1938,
    1950,
    1954,
    1958,
    1962,
    1966,
    1970,
    1974,
    1978,
    1982,
    1986,
    1990,
    1994,
    1998,
    2002,
    2006,
    2010,
    2014
]

assert len(urls) == len(years), "URLs and years must match!"


# =========================
# MAIN LOOP
# =========================
all_rows = []

for url, year in zip(urls, years):
    try:
        response = requests.get(url)
        
        if response.status_code == 200:
            parsed_rows = parse_world_cup_html(response.text, url, year)
            all_rows.extend(parsed_rows)
        else:
            print(f"Failed: {url}")
        
        time.sleep(1)
    
    except Exception as e:
        print(f"Error processing {url}: {e}")


# =========================
# EXPORT
# =========================
df = pd.DataFrame(all_rows)

df.to_csv("world_cup_full_data.csv", index=False)

print("CSV created: world_cup_full_data.csv")

Error processing https://www.rsssf.org/tables/30full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/34full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/38full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/50full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/54full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/58full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/62full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/66full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/70full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/74full.html: name 'line_lower' is not defined
Error processing https://www.rsssf.org/tables/78full.html: name 'line_lower' is 